# Fabric Data Agent Hackathon Evaluation

Enter your observed Data Agent results, run the notebook, and receive a baseline-versus-final scorecard. This notebook does not call the Data Agent API. It loads the six-question challenge definition, evaluates the results you enter, and exports evidence for review.

Each question is worth four points in each phase: correct answer, expected source, correct generated SQL/DAX logic, and paraphrase robustness. Baseline paraphrases are optional and score zero when omitted. Maximum score: **24 baseline** and **24 final**.

## 1. Import Required Libraries

Load the standard Python libraries used to retrieve the challenge definition, validate entries, calculate scores, display tables, and export results. Microsoft Fabric runtimes include pandas and requests. For an ordinary local Jupyter environment, install them first with `pip install pandas requests`.

In [ ]:
import importlib
import json
import math
import re
from datetime import datetime, timezone
from numbers import Number
from pathlib import Path

try:
    pd = importlib.import_module("pandas")
    requests = importlib.import_module("requests")
except ImportError as error:
    raise RuntimeError(
        "This notebook requires pandas and requests. "
        "They are included in Microsoft Fabric; for local Jupyter run: "
        "pip install pandas requests"
    ) from error

try:
    display = importlib.import_module("IPython.display").display
except ImportError:
    display = print

## 2. Configure Evaluation Criteria

Facilitators can adjust the pass threshold and metric weights. The default gives one point each for answer correctness, expected source, inspected query logic, and paraphrase robustness.

In [ ]:
TEAM_NAME = "Enter team name"
AGENT_NAME = "Enter Fabric Data Agent name"

REPOSITORY_OWNER = "Limaoncloud"
REPOSITORY_NAME = "fabric-data-agent-hackathon"
REPOSITORY_REF = "dev"

METRIC_WEIGHTS = {
    "answer": 1.0,
    "source": 1.0,
    "logic": 1.0,
    "paraphrase": 1.0,
}
ANSWER_TOLERANCE = 0.01
PASS_THRESHOLD_PERCENT = 70.0
REQUIRED_TEST_IDS = {f"HC{number:03d}" for number in range(1, 7)}
MAX_SCORE_PER_TEST = sum(METRIC_WEIGHTS.values())
MAX_TOTAL_SCORE = len(REQUIRED_TEST_IDS) * MAX_SCORE_PER_TEST

## 3. Enter Test Results

Edit only the cell below. Answers can be numbers or formatted text such as `£5,420,217`. Enter the Fabric item selected by the agent in each source field. Mark a logic field `True` only after inspecting the generated SQL, DAX, or run steps and confirming the filter and aggregation logic.

Baseline fields may remain blank if no baseline was captured. Every final answer, final source, and final paraphrase answer is required. Entering baseline paraphrase answers allows a like-for-like baseline score out of 24.

In [ ]:
# Enter results from the Fabric Data Agent chat and run details.
TEST_RESULTS = [
    {
        "id": "HC001",
        "baseline_answer": "",
        "baseline_source": "",
        "baseline_logic_correct": False,
        "baseline_paraphrase_answer": "",
        "final_answer": "",
        "final_source": "",
        "final_logic_correct": False,
        "paraphrase_answer": "",
        "notes": "",
    },
    {
        "id": "HC002",
        "baseline_answer": "",
        "baseline_source": "",
        "baseline_logic_correct": False,
        "baseline_paraphrase_answer": "",
        "final_answer": "",
        "final_source": "",
        "final_logic_correct": False,
        "paraphrase_answer": "",
        "notes": "",
    },
    {
        "id": "HC003",
        "baseline_answer": "",
        "baseline_source": "",
        "baseline_logic_correct": False,
        "baseline_paraphrase_answer": "",
        "final_answer": "",
        "final_source": "",
        "final_logic_correct": False,
        "paraphrase_answer": "",
        "notes": "",
    },
    {
        "id": "HC004",
        "baseline_answer": "",
        "baseline_source": "",
        "baseline_logic_correct": False,
        "baseline_paraphrase_answer": "",
        "final_answer": "",
        "final_source": "",
        "final_logic_correct": False,
        "paraphrase_answer": "",
        "notes": "",
    },
    {
        "id": "HC005",
        "baseline_answer": "",
        "baseline_source": "",
        "baseline_logic_correct": False,
        "baseline_paraphrase_answer": "",
        "final_answer": "",
        "final_source": "",
        "final_logic_correct": False,
        "paraphrase_answer": "",
        "notes": "",
    },
    {
        "id": "HC006",
        "baseline_answer": "",
        "baseline_source": "",
        "baseline_logic_correct": False,
        "baseline_paraphrase_answer": "",
        "final_answer": "",
        "final_source": "",
        "final_logic_correct": False,
        "paraphrase_answer": "",
        "notes": "",
    },
]

## 4. Validate Input Data

Load the current six-question challenge from GitHub and check required IDs, duplicate entries, missing fields, Boolean logic values, and configuration ranges. Fix every reported issue before scoring.

In [ ]:
CHALLENGE_URL = (
    f"https://raw.githubusercontent.com/{REPOSITORY_OWNER}/"
    f"{REPOSITORY_NAME}/{REPOSITORY_REF}/evaluation/challenge/uk-legal.json"
 )
response = requests.get(CHALLENGE_URL, timeout=60)
response.raise_for_status()
challenge = response.json()
challenge_by_id = {item["id"]: item for item in challenge["evaluation_queries"]}


def validate_inputs(results, require_final=True):
    errors = []
    ids = [item.get("id") for item in results]
    duplicate_ids = sorted({test_id for test_id in ids if ids.count(test_id) > 1})
    missing_ids = sorted(REQUIRED_TEST_IDS - set(ids))
    unexpected_ids = sorted(set(ids) - REQUIRED_TEST_IDS)

    if duplicate_ids:
        errors.append(f"Duplicate test IDs: {duplicate_ids}")
    if missing_ids:
        errors.append(f"Missing required test IDs: {missing_ids}")
    if unexpected_ids:
        errors.append(f"Unexpected test IDs: {unexpected_ids}")
    if set(challenge_by_id) != REQUIRED_TEST_IDS:
        errors.append("The downloaded challenge IDs do not match REQUIRED_TEST_IDS.")
    if not math.isclose(MAX_SCORE_PER_TEST, 4.0):
        errors.append("Metric weights must total 4.0 points per test.")
    if not 0 <= PASS_THRESHOLD_PERCENT <= 100:
        errors.append("PASS_THRESHOLD_PERCENT must be between 0 and 100.")

    required_fields = {
        "id",
        "baseline_answer",
        "baseline_source",
        "baseline_logic_correct",
        "baseline_paraphrase_answer",
        "final_answer",
        "final_source",
        "final_logic_correct",
        "paraphrase_answer",
        "notes",
    }
    for index, item in enumerate(results, start=1):
        absent = sorted(required_fields - set(item))
        if absent:
            errors.append(f"Entry {index} is missing fields: {absent}")
        for field in ("baseline_logic_correct", "final_logic_correct"):
            if field in item and not isinstance(item[field], bool):
                errors.append(f"{item.get('id', index)}.{field} must be True or False.")
        if require_final:
            blank_fields = [
                field
                for field in ("final_answer", "final_source", "paraphrase_answer")
                if str(item.get(field, "")).strip() == ""
            ]
            if blank_fields:
                errors.append(f"{item.get('id', index)} has blank final fields: {blank_fields}")

    return errors


validation_errors = validate_inputs(TEST_RESULTS, require_final=True)
if validation_errors:
    raise ValueError("Complete or correct the input cell:\n- " + "\n- ".join(validation_errors))

print(f"Validated {len(TEST_RESULTS)} result entries against challenge version {challenge['metadata']['version']}.")

## 5. Calculate Evaluation Metrics

Normalize numeric answers, compare them with the challenge ground truth, check the expected source, apply the configured weights, and calculate baseline and final totals.

In [ ]:
NUMBER_PATTERN = re.compile(r"[-+]?\d[\d,]*(?:\.\d+)?")


def normalize_number(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, Number) and not isinstance(value, bool):
        return float(value)
    matches = NUMBER_PATTERN.findall(str(value).strip())
    if len(matches) != 1:
        return None
    return float(matches[0].replace(",", ""))


def answer_matches(actual, expected):
    actual_number = normalize_number(actual)
    expected_number = normalize_number(expected)
    if actual_number is not None and expected_number is not None:
        return math.isclose(
            actual_number, expected_number, rel_tol=0.0, abs_tol=ANSWER_TOLERANCE
        )
    return str(actual).strip().casefold() == str(expected).strip().casefold()


def source_matches(actual, expected):
    return str(actual).strip().casefold() == str(expected).strip().casefold()


scored_rows = []
for entry in TEST_RESULTS:
    expected = challenge_by_id[entry["id"]]
    baseline_answer_ok = answer_matches(
        entry["baseline_answer"], expected["ground_truth_answer"]
    ) if str(entry["baseline_answer"]).strip() else False
    baseline_source_ok = source_matches(
        entry["baseline_source"], expected["expected_source"]
    ) if str(entry["baseline_source"]).strip() else False
    baseline_logic_ok = entry["baseline_logic_correct"]
    baseline_paraphrase_ok = answer_matches(
        entry["baseline_paraphrase_answer"], expected["ground_truth_answer"]
    ) if str(entry["baseline_paraphrase_answer"]).strip() else False

    final_answer_ok = answer_matches(
        entry["final_answer"], expected["ground_truth_answer"]
    )
    final_source_ok = source_matches(
        entry["final_source"], expected["expected_source"]
    )
    final_logic_ok = entry["final_logic_correct"]
    final_paraphrase_ok = answer_matches(
        entry["paraphrase_answer"], expected["ground_truth_answer"]
    )

    baseline_score = (
        METRIC_WEIGHTS["answer"] * baseline_answer_ok
        + METRIC_WEIGHTS["source"] * baseline_source_ok
        + METRIC_WEIGHTS["logic"] * baseline_logic_ok
        + METRIC_WEIGHTS["paraphrase"] * baseline_paraphrase_ok
    )
    final_score = (
        METRIC_WEIGHTS["answer"] * final_answer_ok
        + METRIC_WEIGHTS["source"] * final_source_ok
        + METRIC_WEIGHTS["logic"] * final_logic_ok
        + METRIC_WEIGHTS["paraphrase"] * final_paraphrase_ok
    )

    scored_rows.append({
        "id": entry["id"],
        "question": expected["question"],
        "paraphrase": expected["paraphrase"],
        "expected_answer": expected["ground_truth_answer"],
        "expected_source": expected["expected_source"],
        "baseline_answer_ok": baseline_answer_ok,
        "baseline_source_ok": baseline_source_ok,
        "baseline_logic_ok": baseline_logic_ok,
        "baseline_paraphrase_ok": baseline_paraphrase_ok,
        "baseline_score": baseline_score,
        "final_answer_ok": final_answer_ok,
        "final_source_ok": final_source_ok,
        "final_logic_ok": final_logic_ok,
        "final_paraphrase_ok": final_paraphrase_ok,
        "final_score": final_score,
        "improvement": final_score - baseline_score,
        "notes": entry["notes"],
    })

results_df = pd.DataFrame(scored_rows)
baseline_total = float(results_df["baseline_score"].sum())
final_total = float(results_df["final_score"].sum())
improvement = final_total - baseline_total
final_percentage = 100.0 * final_total / MAX_TOTAL_SCORE

## 6. Determine Pass Or Fail Status

Apply the configured threshold to each final question and the overall result. Ratings provide a concise interpretation of the final score.

In [ ]:
PER_TEST_PASS_POINTS = 3.0


def rating_for(score):
    if score >= 21:
        return "Strong and robust"
    if score >= 17:
        return "Good, with minor gaps"
    if score >= 12:
        return "Partially improved"
    return "Significant tuning needed"


results_df["status"] = [
    "Pass" if score >= PER_TEST_PASS_POINTS else "Review"
    for score in results_df["final_score"]
]
overall_status = "Pass" if final_percentage >= PASS_THRESHOLD_PERCENT else "Review"
final_rating = rating_for(final_total)
passed_tests = int((results_df["status"] == "Pass").sum())
failed_tests = int((results_df["status"] == "Review").sum())
success_rate = 100.0 * passed_tests / len(results_df)

## 7. Display The Evaluation Report

Review the scorecard, failed checks, and suggested areas before exporting. Keep screenshots or copied SQL/DAX as evidence for the facilitator.

In [ ]:
summary = {
    "team": TEAM_NAME,
    "agent": AGENT_NAME,
    "baseline_score": baseline_total,
    "baseline_max": MAX_TOTAL_SCORE,
    "final_score": final_total,
    "final_max": MAX_TOTAL_SCORE,
    "improvement": improvement,
    "final_percentage": round(final_percentage, 2),
    "overall_status": overall_status,
    "rating": final_rating,
    "passed_tests": passed_tests,
    "tests_for_review": failed_tests,
    "success_rate": round(success_rate, 2),
}

summary_df = pd.DataFrame([summary])
report_columns = [
    "id",
    "question",
    "expected_answer",
    "expected_source",
    "baseline_answer_ok",
    "baseline_source_ok",
    "baseline_logic_ok",
    "baseline_score",
    "final_answer_ok",
    "final_source_ok",
    "final_logic_ok",
    "final_paraphrase_ok",
    "final_score",
    "improvement",
    "status",
]
display(summary_df)
display(results_df[report_columns])

review_rows = results_df.loc[results_df["status"] == "Review"]
if review_rows.empty:
    print("All questions passed the per-test threshold.")
else:
    print("Suggested areas for review:")
    for _, row in review_rows.iterrows():
        missing = []
        if not row["final_answer_ok"]:
            missing.append("answer")
        if not row["final_source_ok"]:
            missing.append("source")
        if not row["final_logic_ok"]:
            missing.append("query/measure logic")
        if not row["final_paraphrase_ok"]:
            missing.append("paraphrase")
        print(f"- {row['id']}: review {', '.join(missing)}")

## 8. Export Evaluation Results

Save the detailed scorecard as CSV and the complete evaluation record as JSON. In Fabric, the files are written to the notebook session's current working directory unless you change `OUTPUT_DIRECTORY`.

In [ ]:
OUTPUT_DIRECTORY = Path(".")
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
safe_team_name = re.sub(r"[^A-Za-z0-9_-]+", "_", TEAM_NAME.strip()).strip("_")
safe_team_name = safe_team_name or "team"
output_stem = f"{safe_team_name}_hackathon_evaluation"
csv_path = OUTPUT_DIRECTORY / f"{output_stem}.csv"
json_path = OUTPUT_DIRECTORY / f"{output_stem}.json"

results_df.to_csv(csv_path, index=False)
export_payload = {
    "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
    "challenge_metadata": challenge["metadata"],
    "evaluation_config": {
        "metric_weights": METRIC_WEIGHTS,
        "answer_tolerance": ANSWER_TOLERANCE,
        "pass_threshold_percent": PASS_THRESHOLD_PERCENT,
        "per_test_pass_points": PER_TEST_PASS_POINTS,
    },
    "summary": summary,
    "results": results_df.to_dict(orient="records"),
}
json_path.write_text(
    json.dumps(export_payload, indent=2, default=str), encoding="utf-8"
 )

print(f"CSV report: {csv_path.resolve()}")
print(f"JSON report: {json_path.resolve()}")

## 9. Verify The Evaluation Logic

Run these assertions after the report. They verify numeric normalization, answer boundaries, validation errors, rating thresholds, and the score ceiling without changing your submitted results.

In [ ]:
assert normalize_number("£5,420,217") == 5420217.0
assert normalize_number("Answer: 54 invoices") == 54.0
assert normalize_number("between 50 and 60") is None
assert answer_matches("£123,590,881", 123590881)
assert answer_matches(101.005, 101.0)
assert not answer_matches(101.02, 101.0)
assert source_matches(" legalfirmsemanticmodel ", "LegalFirmSemanticModel")
assert rating_for(24) == "Strong and robust"
assert rating_for(21) == "Strong and robust"
assert rating_for(20) == "Good, with minor gaps"
assert rating_for(17) == "Good, with minor gaps"
assert rating_for(16) == "Partially improved"
assert rating_for(12) == "Partially improved"
assert rating_for(11) == "Significant tuning needed"

duplicate_sample = [dict(TEST_RESULTS[0]), dict(TEST_RESULTS[0])]
duplicate_errors = validate_inputs(duplicate_sample, require_final=False)
assert any("Duplicate test IDs" in error for error in duplicate_errors)
assert any("Missing required test IDs" in error for error in duplicate_errors)
assert math.isclose(MAX_TOTAL_SCORE, 24.0)
assert final_total <= MAX_TOTAL_SCORE

print("Evaluation logic verification passed.")